# 01 — Data Exploration (Phase 2 EDA)

Run **after** `src/data_collection/youtube_collector.py` has populated `data/raw/youtube/`.

Questions this notebook answers:
- How many videos / comments do we have?
- What is the source distribution?
- Are there obvious spam clusters?
- What is the comment quality like?
- Which products are mentioned most?
- What is the time distribution of content?

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
from pathlib import Path

RAW = Path('../data/raw/youtube')
PROC = Path('../data/processed')

In [ ]:
# Load videos
videos = pd.read_csv(RAW / 'videos.csv', parse_dates=['published_at'])
print(f'Videos: {len(videos)}')
videos.head()

In [ ]:
# Load comments
comments = pd.read_csv(RAW / 'comments.csv', parse_dates=['published_at'])
print(f'Comments: {len(comments)}')
comments.head()

In [ ]:
# Distribution: comments per video
comments_per_video = comments.groupby('video_id').size().sort_values(ascending=False)
px.histogram(comments_per_video, title='Comments per Video', labels={'value': 'Comment count'})

In [ ]:
# Top channels by video count
videos['channel'].value_counts().head(20).plot(kind='barh', title='Videos by Channel')

In [ ]:
# Comment word count distribution
comments['word_count'] = comments['text'].astype(str).str.split().str.len()
comments['word_count'].clip(0, 200).hist(bins=50)
plt.title('Comment word count distribution')
plt.xlabel('Words')
plt.ylabel('Count')
plt.show()
print(comments['word_count'].describe())

In [ ]:
# Run the cleaning pipeline
from src.preprocessing.cleaner import clean_comments_df
comments_clean = clean_comments_df(comments)
print(f'After cleaning: {len(comments_clean)} comments')
PROC.mkdir(exist_ok=True)
comments_clean.to_csv(PROC / 'comments_clean.csv', index=False)
print('Saved → data/processed/comments_clean.csv')

In [ ]:
# Product mention frequency
products = {
    'iPhone Duo':       comments_clean['text_clean'].str.contains('iphone duo', case=False).sum(),
    'iPhone 18 Pro':    comments_clean['text_clean'].str.contains('18 pro', case=False).sum(),
    'iPhone 18 Pro Max':comments_clean['text_clean'].str.contains('pro max', case=False).sum(),
    'Samsung/Android':  comments_clean['text_clean'].str.contains('samsung|android|galaxy', case=False).sum(),
}
pd.Series(products).sort_values().plot(kind='barh', title='Product mention frequency')
plt.tight_layout()
plt.show()